# Sparse Latent Diffusion for Text — Colab Experiments

**Project:** Insert TopK Sparse Autoencoder into Cosmos latent text diffusion pipeline

**Pipeline:**
1. Download pretrained Cosmos checkpoints + ROCStories dataset
2. Extract normalized latents from frozen autoencoder
3. Train SAE (TopK) + Dense AE baseline on cached latents
4. Evaluate: baseline vs SAE-filtered vs Dense AE-filtered generation
5. (Optional) Scenario C: fine-tune diffusion with SAE targets

**Important:** Mount Google Drive to persist results across sessions.

## 0. Setup — Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persistent storage on Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Cosmos_SAE'

import os
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/cached_latents', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/sae_checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/evaluation_results', exist_ok=True)
print("Drive mounted and directories ready.")

In [ ]:
# Clone your fork (replace with your actual repo URL)
REPO_URL = "https://github.com/YOUR_USERNAME/cosmos.git"  # <-- UPDATE THIS

!git clone {REPO_URL} /content/cosmos 2>/dev/null || echo "Repo already cloned"
%cd /content/cosmos

In [ ]:
# Install dependencies
!pip install -q hydra-core omegaconf timm torch-ema optimi rouge-score bert-score mauve-text sacrebleu datasets

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Download Pretrained Checkpoints

Cosmos checkpoints are hosted on AWS S3 (`s3://cosmos-latent-diffusion/checkpoints/`).
You need the autoencoder (trained on Wikipedia) and diffusion model (trained on ROCStories).

**If S3 requires credentials**, download manually and upload to `Drive/Cosmos_SAE/checkpoints/`.

In [ ]:
!pip install -q awscli

# Autoencoder checkpoint
!mkdir -p {DRIVE_PROJECT}/checkpoints/autoencoder-num_latents=16-wikipedia-final-128/
!aws s3 cp s3://cosmos-latent-diffusion/checkpoints/autoencoder-num_latents=16-wikipedia-final-128/100000.pth \
    {DRIVE_PROJECT}/checkpoints/autoencoder-num_latents=16-wikipedia-final-128/100000.pth \
    --region eu-north-1 --no-sign-request

# Diffusion model checkpoint
!mkdir -p {DRIVE_PROJECT}/checkpoints/diffusion-rocstories-16-d=5-final/
!aws s3 cp s3://cosmos-latent-diffusion/checkpoints/diffusion-rocstories-16-d=5-final/180000.pth \
    {DRIVE_PROJECT}/checkpoints/diffusion-rocstories-16-d=5-final/180000.pth \
    --region eu-north-1 --no-sign-request

In [ ]:
# Symlink Drive checkpoints into the repo's working directory
!ln -sfn {DRIVE_PROJECT}/checkpoints /content/cosmos/checkpoints

# Verify checkpoints exist
!ls -lh /content/cosmos/checkpoints/autoencoder-num_latents=16-wikipedia-final-128/
!ls -lh /content/cosmos/checkpoints/diffusion-rocstories-16-d=5-final/

## 2. Download ROCStories Dataset

Pre-processed datasets are on HuggingFace Hub under `bayes-group-diffusion`.
The Cosmos `load_to_hub.py` utility downloads and saves in the expected `.arrow` shard format.

In [ ]:
from datasets import load_dataset
import os

DATA_DIR = "/content/cosmos/data/rocstories"

if not os.path.exists(DATA_DIR):
    print("Downloading ROCStories from HuggingFace Hub...")
    ds = load_dataset("bayes-group-diffusion/rocstories")
    ds.save_to_disk(DATA_DIR)
    print(f"Saved to {DATA_DIR}")
else:
    print(f"Dataset already exists at {DATA_DIR}")

# Verify structure
!ls -la {DATA_DIR}/
!ls {DATA_DIR}/train/ | head -5
!ls {DATA_DIR}/test/ | head -5

## 3. Extract Latents from Frozen Autoencoder

Runs the frozen Cosmos autoencoder over ROCStories, saves normalized latents as `.pt` files.
This decouples SAE training from the autoencoder — only needs to run once.

**Runtime:** ~15-30 min on T4 depending on dataset size.
**Output:** `cached_latents/rocstories/{train,test}/latents_*.pt`

In [ ]:
%cd /content/cosmos

!HYDRA_FULL_ERROR=1 python extract_latents.py \
    dataset=rocstories \
    encoder.latent.num_latents=16 \
    encoder.embedding.max_position_embeddings=128 \
    decoder.latent.num_latents=16 \
    decoder.embedding.max_position_embeddings=128 \
    autoencoder.model.load_checkpoint='"autoencoder-num_latents=16-wikipedia-final-128/100000.pth"'

In [ ]:
# Persist cached latents to Drive (so you never need to re-extract)
!cp -r /content/cosmos/cached_latents {DRIVE_PROJECT}/

# Verify
!ls -lh /content/cosmos/cached_latents/rocstories/train/
!ls -lh /content/cosmos/cached_latents/rocstories/test/

### (Session resume) Restore cached latents from Drive
Run this cell instead of Step 3 if you already extracted latents in a previous session.

In [ ]:
# Only run this if resuming from a previous session
!cp -r {DRIVE_PROJECT}/cached_latents /content/cosmos/
!ls -lh /content/cosmos/cached_latents/rocstories/train/

## 4. Train SAE on Cached Latents

Train TopK SAE and Dense AE baseline on the extracted latents.
This is lightweight — only operates on cached tensors (no autoencoder needed).

**Runtime:** ~20-30 min for 50k steps on T4.
**VRAM:** ~1 GB.

In [ ]:
# 4a. Train TopK SAE (expansion=4x, k=64)
%cd /content/cosmos

!python train_sae.py \
    --latents_dir cached_latents/rocstories \
    --sae_type topk \
    --expansion_factor 4 \
    --k 64 \
    --lr 3e-4 \
    --batch_size 4096 \
    --num_steps 50000 \
    --log_every 100 \
    --eval_every 1000 \
    --checkpoint_every 5000 \
    --output_dir sae_checkpoints/topk_4x_k64

In [ ]:
# 4b. Train Dense AE baseline (same architecture, ReLU instead of TopK)
!python train_sae.py \
    --latents_dir cached_latents/rocstories \
    --sae_type dense \
    --expansion_factor 4 \
    --lr 3e-4 \
    --batch_size 4096 \
    --num_steps 50000 \
    --log_every 100 \
    --eval_every 1000 \
    --checkpoint_every 5000 \
    --output_dir sae_checkpoints/dense_4x

In [ ]:
# Persist SAE checkpoints to Drive
!cp -r /content/cosmos/sae_checkpoints {DRIVE_PROJECT}/
print("SAE checkpoints saved to Drive.")

## 5. Inspect SAE Training Results

Quick sanity check: plot training curves and verify reconstruction quality before evaluation.

In [ ]:
import json
import matplotlib.pyplot as plt

def plot_training_log(log_path, label):
    with open(log_path) as f:
        log = json.load(f)

    steps = [e["step"] for e in log]
    mse = [e["mse"] for e in log]
    fvu = [e["fvu"] for e in log]
    l0 = [e["l0"] for e in log]
    dead = [e["dead_frac_ema"] for e in log]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(label, fontsize=14)

    axes[0, 0].plot(steps, mse)
    axes[0, 0].set_title("MSE Loss")
    axes[0, 0].set_xlabel("Step")

    axes[0, 1].plot(steps, fvu)
    axes[0, 1].set_title("FVU (Fraction of Variance Unexplained)")
    axes[0, 1].set_xlabel("Step")

    axes[1, 0].plot(steps, l0)
    axes[1, 0].set_title("L0 (Active Features)")
    axes[1, 0].set_xlabel("Step")

    axes[1, 1].plot(steps, dead)
    axes[1, 1].set_title("Dead Feature Fraction (EMA)")
    axes[1, 1].set_xlabel("Step")

    plt.tight_layout()
    plt.show()

# Plot both training runs
plot_training_log("sae_checkpoints/topk_4x_k64/training_log.json", "TopK SAE (4x, k=64)")
plot_training_log("sae_checkpoints/dense_4x/training_log.json", "Dense AE (4x, ReLU)")

## 6. Evaluate: Baseline vs SAE vs Dense AE

Runs generation with all three variants and compares MAUVE, diversity, PPL, and rounding error.
Requires the full pipeline (autoencoder + diffusion model).

**Runtime:** ~30-60 min for 2000 texts on T4.
**VRAM:** ~12 GB (tight on T4 — reduce `--num_texts` or `--batch_size` if OOM).

In [ ]:
%cd /content/cosmos

!HYDRA_FULL_ERROR=1 python evaluate_sae.py \
    --dataset rocstories \
    --autoencoder_ckpt "autoencoder-num_latents=16-wikipedia-final-128/100000.pth" \
    --diffusion_ckpt "diffusion-rocstories-16-d=5-final/180000.pth" \
    --sae_ckpt sae_checkpoints/topk_4x_k64/best.pt \
    --dense_ckpt sae_checkpoints/dense_4x/best.pt \
    --sae_type topk \
    --expansion_factor 4 \
    --k 64 \
    --num_texts 2000 \
    --batch_size 32 \
    --output_dir evaluation_results

In [ ]:
# Display results table
import json

with open("evaluation_results/evaluation_results.json") as f:
    results = json.load(f)

# Print formatted comparison
print(f"{'Metric':<25} {'Baseline':>12} {'SAE':>12} {'Dense AE':>12}")
print("-" * 65)
for metric in ["mauve", "diversity", "ppl", "rounding_error_mean", "rounding_error_median"]:
    row = f"{metric:<25}"
    for model in ["baseline", "sae", "dense_ae"]:
        val = results.get(model, {}).get(metric)
        if val is not None and isinstance(val, (int, float)):
            row += f"{val:>12.5f}"
        else:
            row += f"{'N/A':>12}"
    print(row)

In [ ]:
# Persist evaluation results to Drive
!cp -r /content/cosmos/evaluation_results {DRIVE_PROJECT}/
print("Results saved to Drive.")

## 7. (Optional) Scenario C — Fine-tune Diffusion with SAE Targets

Fine-tunes the pretrained diffusion model so it learns to predict SAE-reconstructed
latents directly. This is the strongest integration — the diffusion model's outputs
will naturally land on the SAE's manifold.

**Runtime:** ~1-2 hours for 20k steps on T4.
**VRAM:** ~14 GB (may need A100 on Colab Pro).

In [ ]:
%cd /content/cosmos

!HYDRA_FULL_ERROR=1 python train_diffusion_sae.py \
    dataset=rocstories \
    encoder.latent.num_latents=16 \
    encoder.embedding.max_position_embeddings=128 \
    decoder.latent.num_latents=16 \
    decoder.embedding.max_position_embeddings=128 \
    autoencoder.model.load_checkpoint='"autoencoder-num_latents=16-wikipedia-final-128/100000.pth"' \
    diffusion.model.load_checkpoint='"diffusion-rocstories-16-d=5-final/180000.pth"' \
    diffusion.training.batch_size=64 \
    training=diffusion \
    suffix=sae-finetune \
    +sae_checkpoint=sae_checkpoints/topk_4x_k64/best.pt \
    +sae_type=topk \
    +sae_expansion_factor=4 \
    +sae_k=64 \
    +finetune_lr=1e-5 \
    +finetune_steps=20000

In [ ]:
# Persist fine-tuned diffusion checkpoints to Drive
!cp -r /content/cosmos/checkpoints/diffusion*sae-finetune* {DRIVE_PROJECT}/checkpoints/ 2>/dev/null
print("Fine-tuned checkpoints saved to Drive.")

## 8. SAE Hyperparameter Sweep (Optional)

Try different expansion factors and k values to find the best configuration.

In [ ]:
import subprocess

# Hyperparameter grid
configs = [
    {"expansion_factor": 2, "k": 32,  "name": "topk_2x_k32"},
    {"expansion_factor": 4, "k": 64,  "name": "topk_4x_k64"},   # default
    {"expansion_factor": 4, "k": 128, "name": "topk_4x_k128"},
    {"expansion_factor": 8, "k": 64,  "name": "topk_8x_k64"},
    {"expansion_factor": 8, "k": 128, "name": "topk_8x_k128"},
]

for cfg in configs:
    output_dir = f"sae_checkpoints/{cfg['name']}"
    print(f"\n{'='*60}")
    print(f"Training: {cfg['name']} (expansion={cfg['expansion_factor']}, k={cfg['k']})")
    print(f"{'='*60}")

    cmd = [
        "python", "train_sae.py",
        "--latents_dir", "cached_latents/rocstories",
        "--sae_type", "topk",
        "--expansion_factor", str(cfg["expansion_factor"]),
        "--k", str(cfg["k"]),
        "--lr", "3e-4",
        "--batch_size", "4096",
        "--num_steps", "50000",
        "--output_dir", output_dir,
    ]
    subprocess.run(cmd)

# Copy all results to Drive
!cp -r /content/cosmos/sae_checkpoints {DRIVE_PROJECT}/

In [ ]:
# Compare sweep results — best FVU from each config
import json, os

print(f"{'Config':<25} {'Best FVU':>10} {'Final MSE':>12} {'Final L0':>10}")
print("-" * 60)

for cfg in configs:
    log_path = f"sae_checkpoints/{cfg['name']}/training_log.json"
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        log = json.load(f)
    best_fvu = min(e["fvu"] for e in log)
    final = log[-1]
    print(f"{cfg['name']:<25} {best_fvu:>10.4f} {final['mse']:>12.6f} {final['l0']:>10.1f}")